## Cheung Method


2026/5/13

Extracted SBH by Cheung Methods

Inspired by qwen3.5


1. 物理背景與公式根據熱電子發射理論（Thermionic Emission Theory），流經肖特基接面的電流 $I$
   根據熱電子發射理論，電流公式為：$$I = AA^*T^2 \exp\left( \frac{-q\Phi_b}{kT} \right) \exp\left( \frac{q(V - IR_s)}{nkT} \right)$$當我們對兩邊取對數並整理 $V$ 時，會得到：$$V = IR_s + n\Phi_b + \frac{nkT}{q} \ln\left( \frac{I}{AA^*T^2} \right)$$這時候定義 $H(I)$：$$H(I) \equiv V - \left( \frac{nkT}{q} \right) \ln\left( \frac{I}{AA^*T^2} \right)$$將上面的 $V$ 代入，公式會簡化為：$$H(I) = n\Phi_b + IR_s$$

   其中 $I_s$ 為飽和電流，$R_s$ 是中性區與接觸產生的串聯電阻。當 $V > 3kT/q$ 時，Cheung 提出了兩個函數來線性化處理數據。
---

### Method A, B

I. 函數 A：提取 $R_s$ 與 $n$定義函數 $H(I)$ 如下：$$H(I) = V - \left( \frac{nkT}{q} \right) \ln\left( \frac{I}{AA^*T^2} \right)$$經過推導，可得線性關係式：$$H(I) = n\Phi_b + IR_s$$

II. 函數 B：輔助驗證定義另一個函數 $dV/d(\ln I)$：$$\frac{dV}{d(\ln I)} = \frac{nkT}{q} + IR_s$$

1. 第一步：繪製 $\frac{dV}{d(\ln I)}$ vs. $I$ 圖斜率 (Slope): 代表串聯電阻 $R_s$。截距 (Intercept): 代表 $\frac{nkT}{q}$，由此可計算出理想因子 $n$。這一步不依賴能障高度，因此非常穩定。
2. 第二步：繪製 $H(I)$ vs. $I$ 圖利用第一步得到的 $n$，計算出每個電流點對應的 $H(I)$。斜率 (Slope): 同樣代表 $R_s$（可用來與第一步比對，增加準確性）。截距 (Intercept): 代表 $n\Phi_b$，由此計算出能障高度 $\Phi_b$。

In [ ]:
# %% [markdown]
# # 🔬 Sliding Window Cheung Fitter - 整合測試
# 測試目標：SchottkyAnalyzer → SlidingWindowCheungFitter 呼叫鏈

# %%
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar

# 設置 matplotlib 使用 Arial 字體
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 12,
    'axes.labelsize': 18,
    'axes.titlesize': 18,
    'axes.titleweight': 'bold',
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 20,
    'figure.titleweight': 'bold'
})


from pathlib import Path
import sys

# ✅ 確認 editable install 生效
try:
    from ivs_tools import __version__, HDF5Analyzer
    from ivs_tools.physics.sliding_window_fitter import (
        SlidingWindowCheungFitter, 
        FittingConfig, 
        FittingResult
    )
    
    # 如果 schottky_analyzer 已更新匯出：
    from ivs_tools.physics.schottky_analyzer import SchottkyAnalyzer
    print(f"✅ ivs_tools v{__version__} imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 請執行: pip install -e . 並 restart kernel")
    sys.exit(1)


❌ Import error: No module named 'ivs_tools.physics.cheung_analysis'
💡 請執行: pip install -e . 並 restart kernel


SystemExit: 1

/home/tim/miniconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)



## 1️⃣ 載入測試數據


In [ ]:

# 您的數據路徑
root_path = '/home/tim/Documents/AS/data/nanonis2/2024/20240508/2 um 100 pix bias spec map2'
filename = "2 um 100 pix bias spec map2.h5"
h5map_path = Path(root_path) / filename

# 檢查檔案
if not h5map_path.exists():
    raise FileNotFoundError(f"❌ 找不到檔案: {h5map_path}")

# 初始化 HDF5Analyzer
h5map = HDF5Analyzer(str(h5map_path))
print(f"📁 已載入: {h5map.filepath}")
print(f"📐 空間: {h5map.metadata.x_pixels} × {h5map.metadata.y_pixels} pixels")
print(f"⚡ 偏壓: {len(h5map.bias_vector)} 點, [{h5map.bias_vector.min():.2f}, {h5map.bias_vector.max():.2f}] V")